In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()

In [ ]:
UK_spending_by_country = '''SELECT time_period_value, destination_country, SUM(spend) AS total FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value ORDER BY time_period_value, total DESC'''
df_by_country = bq.read_bq_table_sql(client, UK_spending_by_country)
df_by_country

In [ ]:
UK_spending_by_mccgoods = '''SELECT 
    mcc, 
    SUM(spend) AS total_spend, 
    (SUM(spend) * 100.0 / (SELECT SUM(spend)  
                                          FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
                                          WHERE mcc IN ( 
        'ANTIQUE SHOPS', 
        'ARTIST/CRAFT SHOPS',
        'ART DEALERS & GALLERIES',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'VIDEO GAME ARCADES/ESTABLISH',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY'
) ) ) AS b2b_goods_percentage 
FROM  
    `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
WHERE  
    mcc IN ( 
'ANTIQUE SHOPS', 
        'ARTIST/CRAFT SHOPS',
        'ART DEALERS & GALLERIES',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'VIDEO GAME ARCADES/ESTABLISH',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY') 
and time_period = 'Quarter' 
and merchant_channel = 'Online' 
GROUP BY  
    mcc
ORDER BY  
    total_spend DESC'''
df_by_mccgoods = bq.read_bq_table_sql(client, UK_spending_by_mccgoods)     
df_by_mccgoods

In [ ]:
df_by_mccgoods.to_csv('UK_spending_by_mccgoods.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the data is in the correct format
# Assuming df_by_mccgoods is your DataFrame from the SQL query result

# Sort data by 'total_spend' for better visualization
df_by_mccgoods = df_by_mccgoods.sort_values(by='total_spend', ascending=False)

# Set the plot size for readability
plt.figure(figsize=(12, 8))

# Create a bar chart with total spend
sns.barplot(x='total_spend', y='mcc', data=df_by_mccgoods, palette='viridis')

# Add labels and title
plt.xlabel('Total Spend (£)', fontsize=14)
plt.ylabel('Merchant Category Code (MCC)', fontsize=14)
plt.title('Total Spend by MCC (Online)', fontsize=16)

# Add a second axis to show B2B percentage
# If you want to visualize both total spend and B2B percentage in a dual-axis chart, you can do something like this:

fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot total spend
sns.barplot(x='total_spend', y='mcc', data=df_by_mccgoods, palette='Blues', ax=ax1)
ax1.set_xlabel('Total Spend (£)', fontsize=14)
ax1.set_ylabel('Merchant Category Code (MCC)', fontsize=14)
ax1.set_title('Total Spend and B2B Goods Percentage by MCC (Online Abroad)', fontsize=16)

# Create a second axis for B2B percentage
ax2 = ax1.twiny()
sns.lineplot(x='b2b_goods_percentage', y='mcc', data=df_by_mccgoods, color='red', marker='o', ax=ax2)
ax2.set_xlabel('B2B Goods Percentage (%)', fontsize=14, color='red')
ax2.tick_params(axis='x', labelcolor='red')

# Display the plot
plt.show()

In [ ]:
UK_spending_by_mccgoods_yearly = '''
SELECT 
    CAST(SUBSTR(time_period_value, 1, 4) AS INT64) AS year,  -- Extract the first 4 characters (year) from time_period_value
    SUM(spend) AS mcc_goods_total  -- Sum the total spend for each year
FROM  
    `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
WHERE  
    mcc IN ( 
        'ANTIQUE SHOPS', 
        'ART DEALERS & GALLERIES',
        'ARTIST/CRAFT SHOPS',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'VIDEO GAME ARCADES/ESTABLISH',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY'
    ) 
    AND time_period = 'Quarter' 
    AND merchant_channel = 'Online' 
    AND CAST(SUBSTR(time_period_value, 1, 4) AS INT64) BETWEEN 2019 AND 2024  -- Filter by year range
GROUP BY  
    year
ORDER BY  
    year'''

df_by_mccgoods_yearly = bq.read_bq_table_sql(client, UK_spending_by_mccgoods_yearly)
df_by_mccgoods_yearly


In [ ]:
direct_marketing = '''SELECT time_period_value, destination_country, SUM(spend) AS total FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value ORDER BY time_period_value, total DESC'''